In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))

import pandas as pd

from config import PROCESSED_DATA_DIR

In [2]:
icustays_df = pd.read_parquet(
    PROCESSED_DATA_DIR / "icustays_clean.parquet",
    columns=["stay_id"]
)

chartevents_df = pd.read_parquet(PROCESSED_DATA_DIR / "chartevents_clean.parquet")

print(icustays_df.shape)
print(chartevents_df.shape)
print(chartevents_df["chart_feature"].value_counts())

(94444, 1)
(53537187, 10)
chart_feature
heart_rate     8711106
resp_rate      8568042
spo2           8531645
sbp            8434305
dbp            8432319
mbp            8428002
temperature    2431768
Name: count, dtype: int64


In [3]:
feature_window_hours = 24

chartevents_24h_df = chartevents_df[
    chartevents_df["hours_from_icu_admit"].between(0, feature_window_hours, inclusive="both")
].copy()

print(chartevents_24h_df.shape)
print(chartevents_24h_df["chart_feature"].value_counts())

(15099203, 10)
chart_feature
heart_rate     2444932
resp_rate      2406138
sbp            2393119
dbp            2392483
mbp            2391013
spo2           2388223
temperature     683295
Name: count, dtype: int64


In [4]:
summary_features_df = (
    chartevents_24h_df
    .groupby(["stay_id", "chart_feature"])["valuenum"]
    .agg(["count", "min", "max", "mean", "median", "std", "first", "last"])
    .reset_index()
)

summary_features_df["trend"] = summary_features_df["last"] - summary_features_df["first"]

summary_features_df.head()

,stay_id,chart_feature,count,min,max,mean,median,std,first,last,trend
0,30000153,dbp,34,55.0,118.0,67.794118,66.5,11.267490,77.0,64.0,-13.0
1,30000153,heart_rate,25,83.0,128.0,106.840000,109.0,12.381707,104.0,90.0,-14.0
2,30000153,mbp,33,70.0,121.0,87.060606,87.0,10.289374,84.0,76.0,-8.0
3,30000153,resp_rate,26,10.0,22.0,15.076923,15.0,3.719388,18.0,10.0,-8.0
4,30000153,sbp,34,108.0,171.0,136.088235,134.0,18.375490,113.0,121.0,8.0


In [5]:
chart_features_wide_df = summary_features_df.pivot(
    index="stay_id",
    columns="chart_feature"
)

chart_features_wide_df.columns = [
    f"{feature}_{stat}_24h"
    for stat, feature in chart_features_wide_df.columns
]

chart_features_wide_df = chart_features_wide_df.reset_index()

chart_features_wide_df.head()

,stay_id,dbp_count_24h,heart_rate_count_24h,mbp_count_24h,resp_rate_count_24h,sbp_count_24h,spo2_count_24h,temperature_count_24h,dbp_min_24h,heart_rate_min_24h,...,sbp_last_24h,spo2_last_24h,temperature_last_24h,dbp_trend_24h,heart_rate_trend_24h,mbp_trend_24h,resp_rate_trend_24h,sbp_trend_24h,spo2_trend_24h,temperature_trend_24h
0,30000153,34.0,25.0,33.0,26.0,34.0,25.0,7.0,55.0,83.0,...,121.0,99.0,37.277778,-13.0,-14.0,-8.0,-8.0,8.0,-1.0,0.000000
1,30000213,25.0,25.0,25.0,25.0,25.0,25.0,8.0,47.0,66.0,...,151.0,97.0,37.055556,4.0,17.0,-3.0,-3.0,-17.0,1.0,0.722222
2,30000484,24.0,24.0,24.0,24.0,24.0,24.0,7.0,23.0,81.0,...,119.0,100.0,36.111111,-10.0,-19.0,-3.0,-9.0,18.0,0.0,0.555556
3,30000646,54.0,59.0,54.0,59.0,54.0,59.0,8.0,34.0,69.0,...,82.0,95.0,36.666667,-9.0,-24.0,-12.0,-12.0,-25.0,-2.0,-0.444444
4,30000831,39.0,45.0,39.0,45.0,39.0,45.0,9.0,50.0,80.0,...,103.0,98.0,37.833333,-43.0,-42.0,-36.0,-4.0,-12.0,3.0,0.833333


In [6]:
observed_features_df = (
    chartevents_24h_df
    .groupby(["stay_id", "chart_feature"])
    .size()
    .unstack(fill_value=0)
    .gt(0)
    .astype(int)
)

observed_features_df.columns = [
    f"{feature}_observed_24h"
    for feature in observed_features_df.columns
]

observed_features_df = observed_features_df.reset_index()

observed_features_df.head()

,stay_id,dbp_observed_24h,heart_rate_observed_24h,mbp_observed_24h,resp_rate_observed_24h,sbp_observed_24h,spo2_observed_24h,temperature_observed_24h
0,30000153,1,1,1,1,1,1,1
1,30000213,1,1,1,1,1,1,1
2,30000484,1,1,1,1,1,1,1
3,30000646,1,1,1,1,1,1,1
4,30000831,1,1,1,1,1,1,1


In [7]:
chartevents_features_df = icustays_df.merge(
    chart_features_wide_df,
    on="stay_id",
    how="left"
)

chartevents_features_df = chartevents_features_df.merge(
    observed_features_df,
    on="stay_id",
    how="left"
)

observed_cols = [col for col in chartevents_features_df.columns if col.endswith("_observed_24h")]
chartevents_features_df[observed_cols] = chartevents_features_df[observed_cols].fillna(0).astype(int)

print(chartevents_features_df.shape)
print(chartevents_features_df["stay_id"].duplicated().sum())
print(chartevents_features_df[observed_cols].sum().sort_values(ascending=False))

(94444, 71)
0
heart_rate_observed_24h     94339
spo2_observed_24h           94200
resp_rate_observed_24h      94138
sbp_observed_24h            93911
mbp_observed_24h            93909
dbp_observed_24h            93905
temperature_observed_24h    91594
dtype: int64


In [8]:
chartevents_features_df.to_parquet(
    PROCESSED_DATA_DIR / "chartevents_features.parquet",
    index=False
)